<a href="https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ML-07 setup: load the starter dataset

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/content/flyrank-ML-internship/data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    !git clone -q https://github.com/krihna7/flyrank-ML-internship.git /content/flyrank-ML-internship

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize content for **refresh review** when it is both stale and has meaningful organic visibility.

The baseline uses two signals:

1. **Staleness:** `days_since_last_update`. Older content has a stronger refresh-review signal.
2. **Visibility:** `impressions_90d`. Content with more impressions has a larger potential audience and therefore a larger opportunity if refreshed.

The score combines normalized staleness and normalized visibility. Higher scores mean higher priority for review.

The rule has one action label: **`refresh_review`**.

Reason codes:

* `stale_high_visibility` — the item is relatively stale and has relatively high visibility.
* `stale` — staleness is the main reason for the recommendation.
* `high_visibility` — visibility is the main reason for the recommendation.
* `standard_review` — neither signal is especially strong, but the item remains in the ranked queue.

This is a decision-support baseline, not a claim that a refresh will improve performance.


In [ ]:
# 1A. Signal 1: staleness bucket check

df["staleness_bucket"] = pd.qcut(
    df["days_since_last_update"],
    q=4,
    duplicates="drop"
)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          median_clicks_90d=("clicks_90d", "median")
      )
      .reset_index()
)

print("SIGNAL 1 — days_since_last_update")
display(staleness_table)

print(
    "\nVERDICT: CONFIRMED if the higher-staleness buckets show a "
    "meaningful/directional refresh opportunity; otherwise inspect the table "
    "and record OPPOSITE, MIXED, or FALSE honestly."
)

SIGNAL 1 — days_since_last_update


,staleness_bucket,n,median_impressions_90d,median_clicks_90d
0,"(0.999, 20.0]",15866,363.0,1.0
1,"(20.0, 104.0]",13816,1262.0,1.0
2,"(104.0, 373.0]",318,30.0,0.0



VERDICT: CONFIRMED if the higher-staleness buckets show a meaningful/directional refresh opportunity; otherwise inspect the table and record OPPOSITE, MIXED, or FALSE honestly.


In [ ]:
# 1B. Signal 2: visibility/volume bucket check

df["visibility_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

visibility_table = (
    df.groupby("visibility_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_clicks_90d=("clicks_90d", "median"),
          median_sessions_90d=("sessions_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("SIGNAL 2 — impressions_90d")
display(visibility_table)

print(
    "\nVERDICT: CONFIRMED if higher-impression buckets show greater "
    "opportunity/traffic volume; otherwise inspect the table and record "
    "OPPOSITE, MIXED, or FALSE honestly."
)

SIGNAL 2 — impressions_90d


,visibility_bucket,n,median_clicks_90d,median_sessions_90d,median_ctr
0,"(0.999, 81.0]",7503,0.0,2.0,0.00
1,"(81.0, 731.0]",7499,0.0,4.0,0.00
2,"(731.0, 3615.25]",7498,2.0,11.0,0.13
3,"(3615.25, 517715.0]",7500,22.0,54.0,0.21



VERDICT: CONFIRMED if higher-impression buckets show greater opportunity/traffic volume; otherwise inspect the table and record OPPOSITE, MIXED, or FALSE honestly.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Revised baseline scoring rule

The baseline uses percentile-ranked versions of the two audited signals.

`impressions_90d` receives 70% weight because its signal audit was confirmed. `days_since_last_update` receives 30% because its audit was mixed.

Percentile ranking keeps the relative ordering of observations without allowing extreme values to flatten many pages to the same maximum score.

The final score is used only to prioritize content for refresh review.


In [ ]:
# 2. Build the ranked queue — revised percentile baseline

baseline = df.copy()

# Ensure numeric types
baseline["days_since_last_update"] = pd.to_numeric(
    baseline["days_since_last_update"],
    errors="coerce"
)

baseline["impressions_90d"] = pd.to_numeric(
    baseline["impressions_90d"],
    errors="coerce"
)

# Keep rows with the required observed signals
baseline = baseline.dropna(
    subset=[
        "content_id",
        "days_since_last_update",
        "impressions_90d"
    ]
).copy()

# Non-negative safeguards
baseline["days_since_last_update"] = baseline[
    "days_since_last_update"
].clip(lower=0)

baseline["impressions_90d"] = baseline[
    "impressions_90d"
].clip(lower=0)

# --------------------------------------------------
# Percentile-based signal scores
# --------------------------------------------------

# Confirmed visibility signal: 70%
baseline["visibility_score"] = (
    baseline["impressions_90d"]
    .rank(method="average", pct=True)
)

# Mixed staleness signal: 30%
baseline["staleness_score"] = (
    baseline["days_since_last_update"]
    .rank(method="average", pct=True)
)

# Final baseline score
baseline["baseline_score"] = (
    0.70 * baseline["visibility_score"]
    + 0.30 * baseline["staleness_score"]
)

# --------------------------------------------------
# Reason code
# --------------------------------------------------

baseline["reason_code"] = np.select(
    [
        (baseline["staleness_score"] >= 0.75)
        & (baseline["visibility_score"] >= 0.75),

        baseline["visibility_score"] >= 0.75,

        baseline["staleness_score"] >= 0.75
    ],
    [
        "stale_high_visibility",
        "high_visibility",
        "stale"
    ],
    default="standard_review"
)

# One action label
baseline["action"] = "refresh_review"

# --------------------------------------------------
# Rank
# --------------------------------------------------

baseline = baseline.sort_values(
    [
        "baseline_score",
        "visibility_score",
        "staleness_score"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(
    1,
    len(baseline) + 1
)

# --------------------------------------------------
# Display top 20
# --------------------------------------------------

print("Ranked queue rows:", len(baseline))

display(
    baseline[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "visibility_score",
            "staleness_score"
        ]
    ].head(20)
)

Ranked queue rows: 30000


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,visibility_score,staleness_score
0,1,content_a5dbb404bdc2,0.991240,stale_high_visibility,refresh_review,106,79035,0.991300,0.991100
1,2,content_cf56e2e2e282,0.989430,stale_high_visibility,refresh_review,194,61678,0.986600,0.996033
2,3,content_7368877ea310,0.989150,stale_high_visibility,refresh_review,194,59472,0.986200,0.996033
3,4,content_47b8b12d581e,0.981067,stale_high_visibility,refresh_review,106,40305,0.976767,0.991100
4,5,content_69fad7e6c50c,0.969470,stale_high_visibility,refresh_review,106,28000,0.960200,0.991100
5,6,content_1bfaa38ff26c,0.967730,stale_high_visibility,refresh_review,194,25715,0.955600,0.996033
6,7,content_482aff19e9cc,0.967183,stale_high_visibility,refresh_review,106,26287,0.956933,0.991100
7,8,content_6ac3ab740bbf,0.961362,stale_high_visibility,refresh_review,106,22462,0.948617,0.991100
8,9,content_ac1d924c6a70,0.959717,stale_high_visibility,refresh_review,106,21853,0.946267,0.991100
9,10,content_cb7e312f5d32,0.959220,stale_high_visibility,refresh_review,151,21272,0.944500,0.993567


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 are all assigned the `refresh_review` action because the baseline is designed as a prioritization queue rather than a final decision.

The strongest recommendations combine high observed visibility with relatively high staleness percentile. The main uncertainty is that staleness was only a mixed signal in the audit, so a high score does not prove that a refresh will improve performance.

For each row, the review below records why the item ranked highly and what could make the recommendation wrong.


In [ ]:
# 3. Top-20 review

top20 = baseline.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "visibility_score",
        "staleness_score"
    ]
].copy()


def make_confidence_note(row):
    if (
        row["visibility_score"] >= 0.90
        and row["staleness_score"] >= 0.90
    ):
        return (
            "High baseline evidence: both observed signals "
            "are in high percentile ranges."
        )

    elif row["visibility_score"] >= 0.90:
        return (
            "Moderate-to-high evidence: visibility is very high, "
            "while staleness provides secondary support."
        )

    else:
        return (
            "Moderate evidence: the item ranks highly from the "
            "combined baseline signals."
        )


def make_wrong_condition(row):
    if row["reason_code"] == "stale_high_visibility":
        return (
            "Wrong if the content is intentionally evergreen, "
            "already accurate despite its age, or its recorded "
            "update date does not reflect a recent review."
        )

    elif row["reason_code"] == "high_visibility":
        return (
            "Wrong if high visibility does not represent a useful "
            "refresh opportunity or the content is already performing adequately."
        )

    elif row["reason_code"] == "stale":
        return (
            "Wrong if the content remains accurate despite its age "
            "or the recorded update date understates recent review activity."
        )

    else:
        return (
            "Wrong if the two observed signals do not capture the "
            "actual business value of refreshing the content."
        )


top20_review["confidence_note"] = top20_review.apply(
    make_confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    make_wrong_condition,
    axis=1
)

display(top20_review)

,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,visibility_score,staleness_score,confidence_note,what_would_make_it_wrong
0,1,content_a5dbb404bdc2,0.991240,stale_high_visibility,refresh_review,106,79035,0.991300,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
1,2,content_cf56e2e2e282,0.989430,stale_high_visibility,refresh_review,194,61678,0.986600,0.996033,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
2,3,content_7368877ea310,0.989150,stale_high_visibility,refresh_review,194,59472,0.986200,0.996033,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
3,4,content_47b8b12d581e,0.981067,stale_high_visibility,refresh_review,106,40305,0.976767,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
4,5,content_69fad7e6c50c,0.969470,stale_high_visibility,refresh_review,106,28000,0.960200,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
5,6,content_1bfaa38ff26c,0.967730,stale_high_visibility,refresh_review,194,25715,0.955600,0.996033,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
6,7,content_482aff19e9cc,0.967183,stale_high_visibility,refresh_review,106,26287,0.956933,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
7,8,content_6ac3ab740bbf,0.961362,stale_high_visibility,refresh_review,106,22462,0.948617,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
8,9,content_ac1d924c6a70,0.959717,stale_high_visibility,refresh_review,106,21853,0.946267,0.991100,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...
9,10,content_cb7e312f5d32,0.959220,stale_high_visibility,refresh_review,151,21272,0.944500,0.993567,High baseline evidence: both observed signals ...,Wrong if the content is intentionally evergree...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline has important limitations. A high score can occur because a page has high observed visibility and a relatively high staleness percentile, but that does not prove that refreshing it will improve performance.

Potential weak picks include evergreen content that remains accurate despite its age, content whose recorded update date does not reflect a recent review, and high-visibility content that already performs adequately.

### Leakage check

The score uses only `impressions_90d` and `days_since_last_update`, both observed fields in the available dataset. No future-period performance, target label, or manually assigned outcome is used in the score.

No client names, URLs, private queries, or product flags are used in the ranking.

The result is therefore a baseline decision-support queue, not a prediction of future performance.


In [ ]:
# 4. Weak picks + leakage check

# --------------------------------------------------
# Weak-pick review
# --------------------------------------------------

weak_picks = baseline.tail(10).copy()

print("Weak-pick candidates:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "visibility_score",
            "staleness_score"
        ]
    ]
)


# --------------------------------------------------
# Leakage check
# --------------------------------------------------

score_inputs = {
    "impressions_90d",
    "days_since_last_update"
}

expected_inputs = {
    "impressions_90d",
    "days_since_last_update"
}

print("\nLEAKAGE CHECK")
print("Score inputs:", score_inputs)

assert score_inputs == expected_inputs

print("✓ Only the two intended observed signals are used.")

# Explicitly verify that obvious future/label fields
# are not part of the scoring inputs.
forbidden_terms = [
    "future",
    "label",
    "target",
    "outcome",
    "conversion"
]

for column in score_inputs:
    column_lower = column.lower()

    for term in forbidden_terms:
        assert term not in column_lower

print("✓ No future-window or label-derived score inputs detected.")
print("✓ No product flags used in the score.")

# Verify the action queue exists
assert "action" in baseline.columns
assert "reason_code" in baseline.columns
assert "baseline_score" in baseline.columns

print("✓ Action, reason code, and score are present.")

Weak-pick candidates:


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,visibility_score,staleness_score
29990,29991,content_bb600f317035,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29991,29992,content_92ceb4aee549,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29992,29993,content_994b0a4e4dde,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29993,29994,content_5168e96834b9,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29994,29995,content_b5fb35404aed,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29995,29996,content_8bce3371c63c,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29996,29997,content_2a843f006d86,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29997,29998,content_1d9eca1ce9cd,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29998,29999,content_9ffe1e2e3575,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983
29999,30000,content_0a22a2eeefdd,0.013148,standard_review,refresh_review,1,1,0.017933,0.001983



LEAKAGE CHECK
Score inputs: {'days_since_last_update', 'impressions_90d'}
✓ Only the two intended observed signals are used.
✓ No future-window or label-derived score inputs detected.
✓ No product flags used in the score.
✓ Action, reason code, and score are present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
from pathlib import Path
import pandas as pd

output_path = Path(
    "/content/flyrank-ML-internship/work/outputs/baseline_action_score.csv"
)

# 1. Queue exists
assert len(baseline) == 30000

# 2. Ranking is valid
assert baseline["rank"].iloc[0] == 1
assert baseline["rank"].is_monotonic_increasing

# 3. Scores are valid
assert baseline["baseline_score"].between(0, 1).all()

# 4. Required fields exist
required_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]

for column in required_columns:
    assert column in baseline.columns

# 5. CSV exists
assert output_path.exists()

# 6. CSV can be read
saved_queue = pd.read_csv(output_path)

assert len(saved_queue) == len(baseline)

for column in required_columns:
    assert column in saved_queue.columns

print("======================================")
print("ML-07 SELF-CHECK: PASS")
print("======================================")
print(f"Ranked rows: {len(baseline):,}")
print(f"Top-20 reviewed: {len(top20_review)}")
print(f"CSV exists: {output_path.exists()}")
print("Score inputs: impressions_90d + days_since_last_update")
print("Visibility weight: 70%")
print("Staleness weight: 30%")
print("Future-window inputs: NONE")
print("Label-derived inputs: NONE")
print("Product flags used in score: NONE")
print("======================================")

ML-07 SELF-CHECK: PASS
Ranked rows: 30,000
Top-20 reviewed: 20
CSV exists: True
Score inputs: impressions_90d + days_since_last_update
Visibility weight: 70%
Staleness weight: 30%
Future-window inputs: NONE
Label-derived inputs: NONE
Product flags used in score: NONE
